# 05 — Visualização

Funções de apresentação: grelhas de imagens, tabelas, histogramas, pipelines e overlays.

**Dependências:** `numpy`, `matplotlib`.

**Pré-requisito opcional:** `01-functions.ipynb` (`calcular_histograma_roi`).


## Importações


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline


## Grelha de imagens


In [ ]:
def mostrar_imagens(imagens, titulos=None, mostrar_eixos=False, cmap="gray", max_colunas=4):
    """Apresenta até max_colunas imagens numa única linha."""
    if not isinstance(imagens, (list, tuple)):
        raise ValueError("'imagens' deve ser uma lista ou tuplo.")
    n = len(imagens)
    if n < 1 or n > max_colunas:
        raise ValueError(f"Número de imagens deve estar entre 1 e {max_colunas}.")
    plt.figure(figsize=(5 * n, 5))
    for i, img in enumerate(imagens):
        plt.subplot(1, n, i + 1)
        if img.ndim == 2:
            plt.imshow(img, cmap=cmap)
        else:
            plt.imshow(img)
        if titulos and i < len(titulos):
            plt.title(titulos[i])
        if not mostrar_eixos:
            plt.axis("off")
    plt.tight_layout()
    plt.show()


## Tabelas


In [ ]:
def mostrar_tabela(dados, colunas=None, titulo=None, tamanho=(8, 3), mostrar=True):
    """Tabela matplotlib a partir de lista de linhas."""
    fig, ax = plt.subplots(figsize=tamanho)
    ax.axis("off")
    tabela = ax.table(cellText=dados, colLabels=colunas, loc="center")
    tabela.auto_set_font_size(False)
    tabela.set_fontsize(10)
    tabela.scale(1, 1.5)
    if titulo:
        plt.title(titulo)
    if mostrar:
        plt.show()
    return fig, tabela


## Histogramas


In [ ]:
def mostrar_histograma(img, titulo="Histograma", bins=256, mostrar_media=True):
    """Histograma global da imagem (achatada)."""
    plt.figure(figsize=(6, 4))
    plt.hist(img.ravel(), bins=bins, range=(0, 255), color="gray")
    plt.xlabel("Intensidade")
    plt.ylabel("Frequência")
    plt.title(titulo)
    plt.grid(alpha=0.3)
    if mostrar_media:
        media = float(np.mean(img))
        plt.axvline(media, color="red", linestyle="--", label=f"Média = {media:.1f}")
        plt.legend()
    plt.show()


def mostrar_imagem_e_histograma(img, hist=None, bins=None, titulo_imagem="Imagem", titulo_hist="Histograma"):
    """Duas colunas: imagem | histograma."""
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    cmap = "gray" if img.ndim == 2 else None
    plt.imshow(img, cmap=cmap)
    plt.title(titulo_imagem)
    plt.axis("off")
    plt.subplot(1, 2, 2)
    if hist is None:
        plt.hist(img.ravel(), bins=256, range=(0, 255), color="gray")
    else:
        x = bins[:-1] if bins is not None else np.arange(256)
        plt.bar(x, hist, width=1.0)
    plt.title(titulo_hist)
    plt.xlabel("Intensidade")
    plt.ylabel("Frequência")
    plt.xlim(0, 255)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## Comparação de pipelines


In [ ]:
def mostrar_comparacao_pipeline(imagens, titulos, cmap="gray"):
    """Grelha horizontal para comparar etapas (ex.: original → threshold → morfologia)."""
    mostrar_imagens(list(imagens), titulos=list(titulos), cmap=cmap)


def mostrar_serie_imagem_histograma(imagens, histogramas, titulos):
    """Para cada entrada: imagem (coluna 1) e histograma (coluna 2)."""
    n = len(imagens)
    plt.figure(figsize=(12, 4 * n))
    for i in range(n):
        plt.subplot(n, 2, 2 * i + 1)
        plt.imshow(imagens[i], cmap="gray")
        plt.title(titulos[i])
        plt.axis("off")
        plt.subplot(n, 2, 2 * i + 2)
        plt.bar(np.arange(256), histogramas[i], width=1.0)
        plt.title(f"Histograma — {titulos[i]}")
        plt.xlabel("Intensidade")
        plt.ylabel("Frequência")
        plt.xlim(0, 255)
        plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## Overlays e segmentação


In [ ]:
def mostrar_overlay_mascara(img, mask, alpha=0.45, cor=(1.0, 0.0, 0.0), titulo="Overlay"):
    """Sobrepor máscara binária (bool ou uint8) sobre imagem grayscale/RGB."""
    base = img.copy()
    if base.ndim == 2:
        base_rgb = np.stack([base] * 3, axis=-1)
        if base_rgb.max() <= 1.0:
            base_rgb = (base_rgb * 255).astype(np.uint8)
    else:
        base_rgb = base.copy()
        if base_rgb.max() <= 1.0:
            base_rgb = (base_rgb * 255).astype(np.uint8)
    overlay = base_rgb.astype(np.float64) / 255.0
    m = mask > 0 if mask.dtype != bool else mask
    for c in range(3):
        overlay[:, :, c][m] = (1 - alpha) * overlay[:, :, c][m] + alpha * cor[c]
    plt.figure(figsize=(6, 6))
    plt.imshow(np.clip(overlay, 0, 1))
    plt.title(titulo)
    plt.axis("off")
    plt.show()


def comparar_segmentacao(img, gt, pred, titulos=("Imagem", "Ground truth", "Predição")):
    """Compara imagem de referência, GT e predição lado a lado."""
    mostrar_imagens([img, gt, pred], titulos=list(titulos))
